In [48]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [49]:
load_dotenv()  # Load environment variables from .env file


True

In [50]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [51]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(
        description="The sentiment of the review"
    )

In [52]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [53]:
structured_model=model.with_structured_output(SentimentSchema)
structured_model2=model.with_structured_output(DiagnosisSchema)

In [67]:
model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

prompt = "What is the sentiment of the following review: The software is too good"
structured_model.invoke(prompt).sentiment

'positive'

In [59]:
class ReviewState(TypedDict, total=False):
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [63]:
def find_sentiment(state: ReviewState):
    prompt = f"For the following review, determine the sentiment:\n{state['review']}"
    sentiment = structured_model.invoke(prompt).sentiment
    return {"sentiment": sentiment}


def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state["sentiment"] == "positive":
        return "positive_response"
    return "run_diagnosis"


def positive_response(state: ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:

"{state['review']}"

Also, kindly ask the user to leave feedback on our website."""
    response = model.invoke(prompt).text
    return {"response": response}


def run_diagnosis(state: ReviewState):
    prompt = f"""Diagnose this negative review:

{state['review']}

Return the issue type, emotional tone, and urgency."""
    diagnosis = structured_model2.invoke(prompt)
    return {"diagnosis": diagnosis.model_dump()}


def negative_response(state: ReviewState):
    diagnosis = state["diagnosis"]
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message for this review:

{state['review']}"""
    response = model.invoke(prompt).text
    return {"response": response}

In [64]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges(
    'find_sentiment',
    check_sentiment,
    {
        'positive_response': 'positive_response',
        'run_diagnosis': 'run_diagnosis',
    },
)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [69]:
initial_state = {
    'review': 'I used this product... it kind of good and improved than the recent one and i expect further '
}
workflow.invoke(initial_state)

{'review': 'I used this product... it kind of good and improved than the recent one and i expect further ',
 'sentiment': 'positive',
 'response': 'Here is a warm, appreciative response you can use:\n\n***\n\nHi [Customer Name],\n\nThank you so much for taking the time to share your review! We are really glad to hear that you noticed the improvements in this version compared to the last one. \n\nWe love that you hold us to a high standard—we are always working hard behind the scenes to make our products even better, and feedback like yours keeps us moving in the right direction!\n\nIf you have a quick moment, we would be so grateful if you could share your feedback directly on our website as well [insert link]. Hearing detailed thoughts from customers like you helps our team know exactly what improvements to focus on next.\n\nThanks again for your support, and we look forward to making your next experience even better!\n\nWarm regards,\n\n[Your Name/Company Name]'}